In [ ]:
#r "nuget: Polars.NET, 0.5.0"
#r "nuget: Polars.NET.Core, 0.5.0"
#r "nuget: Polars.NET.Native.linux-x64, 0.5.0"
#r "nuget: Apache.Arrow, 23.0.0"
#r "nuget: Apache.Arrow.Adbc"

using Microsoft.DotNet.Interactive.Formatting;
using Polars.CSharp;
using Pl = Polars.CSharp.Polars;

Formatter.Register<DataFrame>((df,writer)=>
{
    writer.Write(df.ToHtml());
}, "text/html");

Installed Packages Apache.Arrow, 23.0.0 Apache.Arrow.Adbc, 0.23.0

In [9]:
var df = DataFrame.FromColumns(new 
{
    Date = new[] { "2023-01-01", "2023-01-01","2023-01-02" }, // Index 0
    City = new[] { "London", "Manchester","London" },    // Index 1
    Temperature = new[] { 10.5, 9.0,12.1 },   // Index 2
    Rain = new[] {true,true,false} // Index 3
})
.WithColumns(Pl.Col("Date").Str.ToDate("%Y-%m-%d"));

df

Datedate32,Cityutf8view,Temperaturedouble,Rainbool
2023-01-01,"""London""",10.5,true
2023-01-01,"""Manchester""",9,true
2023-01-02,"""London""",12.1,false


In [10]:
var df_converted = df.Filter(Pl.Col("City") == "London")
                    .Select(Pl.Col("Date"),(Pl.Col("Temperature") * 1.8 + 32).Alias("Temp_F"));

df_converted

Datedate32,Temp_Fdouble
2023-01-01,50.9
2023-01-02,53.78


In [11]:
var LondonDf = df.Filter(Pl.Col("City") == "London");

LondonDf

Datedate32,Cityutf8view,Temperaturedouble,Rainbool
2023-01-01,"""London""",10.5,true
2023-01-02,"""London""",12.1,false


In [12]:
var AggDf = df.
        GroupBy(Pl.Col("City"))
        .Agg(Pl.Col("Temperature").Mean().Alias("Avg_Temp")
            ,Pl.Col("Temperature").Max().Alias("Max_Temp")
            ,Pl.Col("Rain").Sum().Alias("Rainy_Days")
            ,Pl.Col("City").Count().Alias("Total_Records"));

AggDf

Cityutf8view,Avg_Tempdouble,Max_Tempdouble,Rainy_Daysuint32,Total_Recordsuint32
"""London""",11.3,12.1,1,2
"""Manchester""",9,9,1,1


In [13]:
var WindowDf = df.Select(Pl.Col("Date"),Pl.Col("City"),Pl.Col("Temperature")
                        ,Pl.Col("Temperature").Mean().Over(Pl.Col("Date")).Alias("Daily_Avg")
                        ,(Pl.Col("Temperature") - Pl.Col("Temperature").Mean().Over(Pl.Col("Date"))).Alias("Diff")
        ).Sort("Date",false);

WindowDf

Datedate32,Cityutf8view,Temperaturedouble,Daily_Avgdouble,Diffdouble
2023-01-01,"""London""",10.5,9.75,0.75
2023-01-01,"""Manchester""",9,9.75,-0.75
2023-01-02,"""London""",12.1,12.1,0


In [14]:
var lf = 
    df.Lazy()
        .Filter(Pl.Col("Temperature") > 10.0)
        .GroupBy("City")
        .Agg(Pl.Col("Temperature").Mean().Alias("Lazy_Avg_Temp"));

System.Console.WriteLine(lf.Explain(true));

var finalDf = lf.Collect();

finalDf

AGGREGATE[maintain_order: true]
  [col("Temperature").mean().alias("Lazy_Avg_Temp")] BY [col("City")]
  FROM
  FILTER [(col("Temperature")) > (10.0)]
  FROM
    DF ["Date", "City", "Temperature", "Rain"]; PROJECT["Temperature", "City"] 2/4 COLUMNS


Cityutf8view,Lazy_Avg_Tempdouble
"""London""",11.3
